In [ ]:
# Cell 0 · Install & Import
!pip install awswrangler tensorflow --quiet
import awswrangler as wr
import numpy as np
import pandas as pd
import pickle
import warnings
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
warnings.filterwarnings('ignore')
print('OK')

In [ ]:
# Cell 1 · Config
S3_BUCKET  = 'gold-lstm-forecast'
MODEL_PATH = f's3://{S3_BUCKET}/gold/xauusd_daily/features'
print(f'Model path : {MODEL_PATH}')

In [ ]:
# Cell 2 · Load Data
for fname in ['X_train.npy','X_test.npy','y_train.npy','y_test.npy',
              'feature_scaler.pkl','target_scaler.pkl']:
    wr.s3.download(path=f'{MODEL_PATH}/{fname}', local_file=f'/tmp/{fname}')

X_train = np.load('/tmp/X_train.npy')
X_test  = np.load('/tmp/X_test.npy')
y_train = np.load('/tmp/y_train.npy')
y_test  = np.load('/tmp/y_test.npy')

with open('/tmp/target_scaler.pkl', 'rb') as f:
    target_scaler = pickle.load(f)

print('=== Data Loaded ===')
print(f'  X_train : {X_train.shape}')
print(f'  X_test  : {X_test.shape}')

In [ ]:
# Cell 3 · Walk-Forward Retraining (builds + trains the real model)
SEQ_LEN       = 60
RETRAIN_EVERY = 7

X_train_scaled = np.load('/tmp/X_train.npy')
X_test_scaled  = np.load('/tmp/X_test.npy')
y_train_scaled = np.load('/tmp/y_train.npy')
y_test_scaled  = np.load('/tmp/y_test.npy')

with open('/tmp/target_scaler.pkl', 'rb') as f:
    target_scaler = pickle.load(f)

print(f'X_train: {X_train_scaled.shape} | X_test: {X_test_scaled.shape}')

all_X_flat = np.vstack([
    X_train_scaled[:, -1, :],
    X_test_scaled[:, -1, :]
])
all_y_scaled = np.concatenate([y_train_scaled, y_test_scaled])

train_size = len(X_train_scaled)
test_size  = len(X_test_scaled)

print(f'all_X_flat  : {all_X_flat.shape}')
print(f'all_y_scaled: {all_y_scaled.shape}')


def build_model():
    m = Sequential([
        LSTM(64, return_sequences=True, input_shape=(SEQ_LEN, 8)),
        Dropout(0.1),
        LSTM(32),
        Dropout(0.1),
        Dense(1)
    ])
    m.compile(optimizer='adam', loss='huber')
    return m


def create_sequences(X_flat, y, seq_len):
    # Include day i itself in the window (see notebook 04 fix) so the model
    # sees the most recent known day before predicting next-day close.
    Xs, ys = [], []
    for i in range(seq_len - 1, len(X_flat)):
        Xs.append(X_flat[i-seq_len+1:i+1])
        ys.append(y[i])
    return np.array(Xs), np.array(ys)


wf_predictions = []
wf_actuals     = []

step = 0
while True:
    test_start = train_size + (step * RETRAIN_EVERY)
    test_end   = min(test_start + RETRAIN_EVERY, train_size + test_size)

    if test_start >= train_size + test_size:
        break

    print(f'[Step {step+1}] Train: {max(0,test_start-365)}->{test_start} | Test: {test_start}->{test_end}')

    X_tr = all_X_flat[max(0, test_start-365):test_start]
    y_tr = all_y_scaled[max(0, test_start-365):test_start]

    X_seq, y_seq = create_sequences(X_tr, y_tr, SEQ_LEN)

    # Chronological validation split (last 15% of the training window) so
    # early stopping actually guards against overfitting instead of just
    # watching training loss converge.
    n_val = max(1, int(len(X_seq) * 0.15))
    X_fit, y_fit = X_seq[:-n_val], y_seq[:-n_val]
    X_val, y_val = X_seq[-n_val:], y_seq[-n_val:]

    m = build_model()
    early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
    m.fit(X_fit, y_fit, validation_data=(X_val, y_val), epochs=20, batch_size=64,
          callbacks=[early_stop], verbose=0)

    for i in range(test_start, test_end):
        if i < SEQ_LEN:
            continue
        x_input = all_X_flat[i-SEQ_LEN+1:i+1].reshape(1, SEQ_LEN, 8)
        pred    = m.predict(x_input, verbose=0)[0][0]
        wf_predictions.append(pred)
        wf_actuals.append(all_y_scaled[i])

    step += 1

wf_pred   = target_scaler.inverse_transform(
    np.array(wf_predictions).reshape(-1, 1)).ravel()
wf_actual = target_scaler.inverse_transform(
    np.array(wf_actuals).reshape(-1, 1)).ravel()

print(f'\nWalk-forward done | {len(wf_pred)} predictions')

np.save('/tmp/wf_pred.npy',   wf_pred)
np.save('/tmp/wf_actual.npy', wf_actual)
print('Saved -> /tmp/wf_pred.npy  &  /tmp/wf_actual.npy')

In [ ]:
# Cell 4 · Save Model
m.save('/tmp/lstm_model.keras')
wr.s3.upload(local_file='/tmp/lstm_model.keras',
             path=f'{MODEL_PATH}/lstm_model.keras')
wr.s3.upload(local_file='/tmp/wf_pred.npy',
             path=f'{MODEL_PATH}/wf_pred.npy')
wr.s3.upload(local_file='/tmp/wf_actual.npy',
             path=f'{MODEL_PATH}/wf_actual.npy')
print('All saved to S3')
print()
print('Notebook 05 DONE')
print('Next -> 06_Model_Evaluation.ipynb')